# Where insiders bought their own stock in 2026

Every open-market Form 4 purchase of $100,000 or more from January to September 2026: who bought, in which months, and the companies where five or more insiders bought.

Read it with charts at https://www.quantisnow.com/sec-api/notebooks/insider-buying-2026. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Executives sell for a hundred reasons: taxes, a house, a diversification plan set a year ago. They buy with their own cash for one. This notebook pulls every open-market purchase of $100,000 or more filed on Form 4 from January through September 2026 and asks where the buying piled up.

The helper pages through a query 50 records at a time, one API call per page. Every notebook here starts with it.

In [1]:
import os, requests
import pandas as pd

API = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(dataset, query, start=0, size=50, sort=None):
    body = {"query": query, "from": start, "size": size}
    if sort:
        body["sort"] = sort
    r = requests.post(f"{API}/{dataset}", json=body,
                      headers={"Authorization": f"Bearer {KEY}"})
    r.raise_for_status()
    return r.json()

def search_all(dataset, query, sort=None):
    """Every record that matches, 50 per call."""
    rows, start = [], 0
    while True:
        page = search(dataset, query, start, 50, sort)
        rows += page["data"]
        start += 50
        if start >= min(page["total"]["value"], 10_000):
            return rows

## Pull the purchases

Transaction code `P` is an open-market purchase. `restatesEarlier:false` drops amendments that repeat a trade already filed, so each purchase counts once.

In [2]:
QUERY = ("formType:4 AND transactionCodes:P AND purchased.valueUsd:>=100000"
         " AND restatesEarlier:false AND filedAt:[2026-01-01 TO 2026-09-30]")
buys = pd.json_normalize(search_all("insider-trades", QUERY))
print(f"{len(buys):,} purchases, ${buys['purchased.valueUsd'].sum() / 1e9:.1f}B")

3,059 purchases, $13.4B

## Officers, directors and big holders

A fund that owns 10% of a company files Form 4 like an insider does, and its purchases are often the largest. Splitting by role keeps them from drowning out the executives.

In [3]:
def role(r):
    if r["reportingOwner.isOfficer"]:
        return "officer"
    if r["reportingOwner.isDirector"]:
        return "director"
    return "10% owner only"

buys["role"] = buys.apply(role, axis=1)
buys.groupby("role")["purchased.valueUsd"].agg(["count", "sum"])

role,count,sum
director,"1,299","4,101,616,118"
officer,"1,038","2,409,690,298"
10% owner only,722,"6,918,224,016"


Big holders put in $6.9 billion of the $13.4 billion, in 722 purchases. Directors bought more than officers: $4.1 billion against $2.4 billion. The rest of the notebook keeps only officers and directors.

## Month by month

In [4]:
insiders = buys[buys["role"] != "10% owner only"].copy()
insiders["month"] = insiders["filedAt"].str[:7]
monthly = insiders.groupby("month")["purchased.valueUsd"].sum()
monthly.plot.bar(title="Officer and director purchases by month, 2026")

Officer and director purchases by month, 2026
Jan	203731564
Feb	499175134
Mar	1105968470
Apr	443520432
May	1163104465
Jun	647239238
Jul	1345902569
Aug	495028152
Sep	607636392

March, May and July were the heaviest months, each over $1.1 billion. January was the quietest at about $200 million, which fits the blackout windows most companies impose before fourth-quarter results.

## Clusters: five or more insiders at one company

One director buying is a data point. Five different insiders at the same company in the same year is a pattern. Count people by `reportingOwner.cik`, not by name, since names are spelled differently from one form to the next.

In [5]:
insiders["ceo"] = insiders["reportingOwner.officerTitle"].fillna("").str.contains("chief executive|ceo", case=False)
clusters = (insiders.groupby(["issuer.ticker", "issuer.name"])
    .agg(insiders=("reportingOwner.cik", "nunique"),
         usd=("purchased.valueUsd", "sum"),
         ceo_bought=("ceo", "any"))
    .query("insiders >= 5")
    .sort_values(["insiders", "usd"], ascending=False))
clusters.head(15)

issuer.ticker,issuer.name,insiders,usd,ceo_bought
BBDO,Bank Bradesco,22,"29,961,051",False
AMRZ,Amrize Ltd,11,"9,449,987",True
NCLH,Norwegian Cruise Line Holdings Ltd.,7,"29,163,646",True
CDNL,Cardinal Infrastructure Group Inc.,7,"9,818,614",True
ARXS,"Arxis, Inc.",7,"5,643,792",False
FISV,Fiserv Inc,7,"2,242,901",False
HTGC,"Hercules Capital, Inc.",7,"2,029,266",True
HLNE,Hamilton Lane INC,6,"18,369,320",True
GO,Grocery Outlet Holding Corp.,6,"8,098,561",True
GEHC,GE HealthCare Technologies Inc.,6,"6,298,440",True


Thirty companies had five or more insiders buying. The top of the list is a surprise: 22 officers and directors of Brazil's Bank Bradesco reported purchases of its preference shares on Form 4, about $30 million in all. Further down, Norwegian Cruise Line had seven insiders including the CEO, and SpyGlass Pharma had five insiders putting in $84 million between them.

## The largest CEO purchases

In [6]:
cols = ["filedAt", "issuer.ticker", "reportingOwner.name", "purchased.valueUsd"]
top = insiders[insiders["ceo"]].nlargest(10, "purchased.valueUsd")[cols]
top.assign(filedAt=top["filedAt"].str[:10])

filedAt,issuer.ticker,reportingOwner.name,purchased.valueUsd
2026-03-04,TTD,Green Jeffrey Terry,"148,101,266"
2026-06-12,SMMT,Zanganeh Mahkam,"49,987,200"
2026-06-12,SMMT,DUGGAN ROBERT W,"49,987,200"
2026-06-22,CNTM,Panigrahi Bhaskar,"40,907,083"
2026-04-13,COE,Huang Jack Jiajia,"35,332,457"
2026-09-22,GRAB,Tan Anthony Ping Yeow,"29,876,310"
2026-09-21,GME,Cohen Ryan,"26,393,723"
2026-05-21,CSAN,Martins Marcelo Eduardo,"23,820,132"
2026-07-08,COE,Huang Jack Jiajia,"23,487,344"
2026-01-22,GME,Cohen Ryan,"21,359,200"


The Trade Desk's Jeff Green made the largest CEO purchase, $148 million in March. Summit Therapeutics shows up twice with the same $49,987,200 because its two co-CEOs reported one purchase on a joint filing. To count it once, drop duplicates on `accessionNo`. GameStop's Ryan Cohen and 51Talk's Jack Huang each appear twice, buying in two separate months.

## Take it further

Change the window or lower the $100,000 floor in `QUERY`. To check one company over several years, add `AND issuer.ticker:NCLH`. Each insider's purchases link to the filing through `accessionNo`, and the [Form 4 dataset page](https://www.quantisnow.com/sec-api/insider-trading) lists every field.